In [9]:
import pandas as pd
import geopandas as gpd
import numpy as np

import matplotlib.pyplot as plt

import os

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, BooleanType, DateType, MapType, FloatType, ArrayType
from pyspark.sql.functions import from_json, get_json_object, col, unix_timestamp, substring, udf, floor, collect_list
from pyspark.sql.functions import max as pyspark_max
from pyspark.sql.functions import min as pyspark_min


from pyproj import Transformer
# transformer from lat-lon system to web mercator system (no the best but will do for now)
transformer = Transformer.from_crs(4326, 3857)

from ast import literal_eval

from shapely.geometry import Point
from tqdm import trange
import contextily as cx
import infostop

In [7]:
#functions to extract lat/lon/x/y from tweets

@udf(returnType=FloatType())
def get_xcoord(x,y):
    return transformer.transform(x,y)[0]

@udf(returnType=FloatType())
def get_ycoord(x,y):
    return transformer.transform(x,y)[1]

@udf(returnType=FloatType())
def get_lat(s):
    if s is not None:
        return float(literal_eval(s)[1])
    else:
        return None

@udf(returnType=FloatType())
def get_lon(s):
    if s is not None:
        return float(literal_eval(s)[0])
    else:
        return None

In [3]:
# initializing Spark session
spark = SparkSession \
    .builder\
    .config("spark.driver.memory", "50g")\
    .appName("Python Spark SQL") \
    .getOrCreate()

25/10/21 13:10:40 WARN Utils: Your hostname, ab-vsrv09 resolves to a loopback address: 127.0.1.1; using 10.50.152.25 instead (on interface ens160)
25/10/21 13:10:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/10/21 13:10:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Defining data types for columns.

In [4]:
# structure of tweets saved by Bence after pre-processing
tweets_schema = StructType(
    [
    StructField("attachments", StringType(), True),
    StructField("author_created_at", StringType(), True),
    StructField("author_description", StringType(), True),
    StructField("author_entitites", StringType(), True),
    StructField("author_id", LongType(), True),
    StructField("author_location", StringType(), True),
    StructField("author_name", StringType(), True),
    StructField("author_pinned_tweet_id", LongType(), True),
    StructField("author_pm_followers_count", LongType(), True),
    StructField("author_pm_following_count", LongType(), True),
    StructField("author_pm_listed_count", LongType(), True),
    StructField("author_pm_tweet_count", LongType(), True),
    StructField("author_profile_image_url", StringType(), True),
    StructField("author_protected", BooleanType(), True),
    StructField("author_url", StringType(), True),
    StructField("author_username", StringType(), True),
    StructField("author_verified", BooleanType(), True),
    StructField("author_withheld", StringType(), False),
    StructField("context_annotations", StringType(), True),
    StructField("conversation_id", LongType(), True),
    StructField("created_at", StringType(), False),
    StructField("edit_controls", StringType(), False),
    StructField("edit_history_tweet_ids", StringType(), True),
    StructField("entities", StringType(), True),
    StructField("geo_coo_coordinates", StringType(), True),
    StructField("geo_coo_type", StringType(), True),
    StructField("geo_loc_name", StringType(), True),
    StructField("geo_place_id", StringType(), True),
    StructField("id", LongType(), False),
    StructField("in_reply_to_user_id", LongType(), True),
    StructField("lang", StringType(), True),
    StructField("possibly_sensitive", BooleanType(), True),
    StructField("referenced_tweets", StringType(), True),
    StructField("reply_settings", StringType(), True),
    StructField("source", StringType(), True),    
    StructField("text", StringType(), False),
    StructField("tweet_pm_like_count", LongType(), False),
    StructField("tweet_pm_quote_count", LongType(), False),
    StructField("tweet_pm_reply_count", LongType(), False),
    StructField("tweet_pm_retweet_count", LongType(), False),
    StructField("withheld", StringType(), False)
    ]
)

Options to read tweet texts correctly.

Users to include:
* at least 10 tweets
* at least 10 different days in the data
* at least 1 months max - min time
* varied text
* high temporal diversity
* good enough following / follower ratio

In [ ]:
for city in ['amsterdam', 'portland']:
    tweets = spark.read\
        .option("multiline", "true")\
        .option("quote", '"')\
        .option("escape", "\\")\
        .option("escape", '"')\
        .csv(
            f'../data/{city}/tweets/',
            header="True",
            schema=tweets_schema
        )\
        .select(
            col("id"),
            col("author_id"),
            col("author_username"),
            col("text"),
            substring(col("created_at"),1,19).alias("created_at"),
            col("author_pm_followers_count"),
            col("author_pm_following_count"),
            col("author_pm_tweet_count"),
            col("geo_coo_coordinates"),
            col("geo_place_id"),
        )\
        .withColumn("timestamp",unix_timestamp("created_at"))\
        .withColumn("date",substring(col("created_at"),1,10))\
        .withColumn("hour",substring(col("created_at"),12,2))\
        .withColumn("lat",get_lat(col("geo_coo_coordinates")))\
        .withColumn("lon",get_lon(col("geo_coo_coordinates")))\
        .filter(col("lat").isNotNull())\
        .filter(col("lon").isNotNull())\
        .withColumn("x_trunc",floor(get_xcoord(col("lon"),col("lat"))/1000)*1000)\
        .withColumn("y_trunc",floor(get_ycoord(col("lon"),col("lat"))/1000)*1000)
    
    users_group_min_10_tweets = tweets\
        .groupBy("author_id")\
        .count()\
        .select(col("author_id"),col("count").alias("tweet_count"))\
        .filter("tweet_count>10")\
        .toPandas()

    users_group_min_10_days = tweets\
        .groupby("author_id","date")\
        .count()\
        .groupby("author_id")\
        .count()\
        .select(col("author_id"),col("count").alias("unique_days"))\
        .filter("unique_days>10")\
        .toPandas()

    users_group_num_grid_cells = tweets\
        .groupby("author_id","x_trunc","y_trunc")\
        .count()\
        .groupby("author_id")\
        .count()\
        .select(col("author_id"),col("count").alias("unique_grid_cells"))\
        .toPandas()

    users_group_min_1_month = tweets\
        .groupby("author_id")\
        .agg(
            pyspark_min("timestamp").alias("min_ts"),
            pyspark_max("timestamp").alias("max_ts"),
        )\
        .filter("(max_ts-min_ts)>2678400")\
        .toPandas()

    users_group_ff = tweets\
        .groupby("author_id")\
        .agg(
            pyspark_max("author_pm_followers_count").alias("followers"),
            pyspark_max("author_pm_following_count").alias("following"),
        ).toPandas()

    users_group_min_10_days.set_index("author_id",inplace=True)
    users_group_min_10_tweets.set_index("author_id",inplace=True)
    users_group_min_1_month.set_index("author_id",inplace=True)
    users_group_num_grid_cells.set_index("author_id",inplace=True)
    users_group_ff.set_index("author_id",inplace=True)

    selected_users = \
        users_group_min_10_days\
            .join(users_group_min_10_tweets)\
            .join(users_group_min_1_month)\
            .join(users_group_num_grid_cells)\
            .join(users_group_ff)
    
    selected_users.reset_index(inplace=True)

    f = (selected_users["followers"]<5000)&\
        (selected_users["following"]<5000)&\
        (selected_users["following"]>10)#&\
        # (selected_users["unique_grid_cells"]>1)

    print("Number of selected users in",city, ":",f.sum())

    selected_users[f].to_csv(f'selected_users_{city}.csv',index=False,header=True)